# 🎧 简化版流程：补齐人声并导出
- 仅保留最后阶段所需的三个步骤。
- 默认读取已经准备好的干净人声、伴奏与原始视频。
- 运行顺序：Cell 1 ➜ Cell 2 ➜ Cell 3 ➜ Cell 4。

In [ ]:
# Cell 1: 基础配置
from pathlib import Path
from datetime import datetime

# 手动调整下面的路径以匹配你的文件位置
OUTPUT_DIR = Path(r"output_project")
MANUAL_VOCAL_PATH = OUTPUT_DIR / "out_2_MixDown.wav"  # 需补齐静音的人声
BACKGROUND_PATH = OUTPUT_DIR / "htdemucs" / "beautifulmyth_audio" / "other.wav"
VIDEO_PATH = Path(r"INPUT_VIDEO.mp4")

PADDED_VOCAL_PATH = OUTPUT_DIR / "final_vocal_padded.wav"
MIX_OUTPUT_PATH = OUTPUT_DIR / "final_mix_output_for_video.wav"
FINAL_VIDEO_PATH = OUTPUT_DIR / "final_video_result.mp4"

for label, path in {
    "输出目录": OUTPUT_DIR,
    "人声": MANUAL_VOCAL_PATH,
    "伴奏": BACKGROUND_PATH,
    "原视频": VIDEO_PATH
}.items():
    if not path.exists():
        print(f"⚠️ {label} 未找到: {path}")
    else:
        print(f"✅ {label}: {path}")

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\n配置完成（{timestamp}）。按顺序继续运行后续单元格。")

In [ ]:
# Cell 2: 人声补齐到伴奏时长
from pydub import AudioSegment

if not MANUAL_VOCAL_PATH.exists() or not BACKGROUND_PATH.exists():
    raise FileNotFoundError("请确认人声和伴奏路径正确后再运行本单元格。")

vocal = AudioSegment.from_file(MANUAL_VOCAL_PATH)
backing = AudioSegment.from_file(BACKGROUND_PATH)

# 统一采样率与声道数量避免导出时出现失真
if vocal.frame_rate != backing.frame_rate:
    vocal = vocal.set_frame_rate(backing.frame_rate)
if vocal.channels != backing.channels:
    vocal = vocal.set_channels(backing.channels)

diff_ms = len(backing) - len(vocal)
if diff_ms > 0:
    padded_vocal = vocal + AudioSegment.silent(duration=diff_ms, frame_rate=backing.frame_rate)
    action = f"尾部补齐 {diff_ms/1000:.2f} 秒静音"
elif diff_ms < 0:
    padded_vocal = vocal[:len(backing)]
    action = f"截去超出部分 {abs(diff_ms)/1000:.2f} 秒"
else:
    padded_vocal = vocal
    action = "长度一致，无需调整"

padded_vocal.export(PADDED_VOCAL_PATH, format="wav")
print(f"人声长度: {len(vocal)/1000:.2f}s ➜ {len(padded_vocal)/1000:.2f}s；{action}。")
print(f"已导出补齐后人声: {PADDED_VOCAL_PATH}")

In [ ]:
# Cell 3: 混音导出最终音频
from pydub import AudioSegment

if not PADDED_VOCAL_PATH.exists():
    raise FileNotFoundError("未找到补齐后人声，请先执行上一单元格。")
if not BACKGROUND_PATH.exists():
    raise FileNotFoundError("未找到伴奏文件，请检查路径设置。")

padded_vocal = AudioSegment.from_file(PADDED_VOCAL_PATH)
backing = AudioSegment.from_file(BACKGROUND_PATH)

# 统一参数并叠加音轨
if padded_vocal.frame_rate != backing.frame_rate:
    padded_vocal = padded_vocal.set_frame_rate(backing.frame_rate)
if padded_vocal.channels != backing.channels:
    padded_vocal = padded_vocal.set_channels(backing.channels)

final_mix = (backing - 12).overlay(padded_vocal + 4)
final_mix.export(MIX_OUTPUT_PATH, format="wav")

print(f"混音完成，导出文件: {MIX_OUTPUT_PATH}")
print(f"最终音频时长: {len(final_mix)/1000:.2f}s")

In [ ]:
# Cell 4: 替换原视频音频
import subprocess

if not VIDEO_PATH.exists():
    raise FileNotFoundError("未找到原始视频，请在 Cell 1 中调整 VIDEO_PATH。")
if not MIX_OUTPUT_PATH.exists():
    raise FileNotFoundError("未找到最终音频，请先完成混音步骤。")

cmd = [
    "ffmpeg","-y",
    "-i", str(VIDEO_PATH),
    "-i", str(MIX_OUTPUT_PATH),
    "-c:v", "copy",
    "-c:a", "aac",
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-shortest",
    str(FINAL_VIDEO_PATH)
]

print("开始合成最终视频...\n"
      f"视频源: {VIDEO_PATH}\n"
      f"音频源: {MIX_OUTPUT_PATH}\n"
      f"输出: {FINAL_VIDEO_PATH}")

subprocess.run(cmd, check=True)
print("✅ 视频合成完成。")